In [143]:
import numpy as np
import sys
import time
import h5py
from tqdm import tqdm
import pyfastx
import numpy as np
import re
from math import ceil
from sklearn.metrics import average_precision_score
from torch.utils.data import Dataset
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import pickle
#import pickle5 as pickle
import os
from sklearn.model_selection import train_test_split

from scipy.sparse import load_npz
from glob import glob

from transformers import get_constant_schedule_with_warmup
from sklearn.metrics import precision_score,recall_score,accuracy_score

from src.train import trainModel
from src.dataloader import getData,spliceDataset,h5pyDataset,collate_fn, getDataPointListFull
from src.weight_init import keras_init
from src.losses import categorical_crossentropy_2d
from src.model import SpliceFormer, SpliceAI_10K
from src.evaluation_metrics import print_topl_statistics
import copy
from collections import defaultdict

In [2]:
data_dir = '../Data'
SL=5000
CL_max=40000
setType = 'test'
annotation_test, transcriptToLabel_test, seqData = getData(data_dir, setType)
BATCH_SIZE = 1
specific_anno = annotation_test[annotation_test['name'].apply(lambda x: x.split('--')[0])=='CFTR']

In [3]:
path = '../Results/Grad-CAM/CFTR_Transformer45k_skips.pkl'

In [5]:
CL_max = 40000
SL = 5000
device = torch.device('cpu')
test_dataset = spliceDataset(getDataPointListFull(specific_anno,transcriptToLabel_test,SL,CL_max,shift=SL))
test_dataset.seqData = seqData
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
for i,(batch_chunks,target_chunks) in enumerate(tqdm(test_loader)):
    batch_features = batch_chunks.to(torch.float32).to(device)      # shape: [1, 4, 45000]
    targets = torch.squeeze(target_chunks.to(torch.float32).to(device), dim=0)  # shape: [3, 45000]
    
    signal = batch_features[0].sum(dim=0)  # shape: [45000]
    nonzero_positions = torch.nonzero(signal > 0).squeeze()
    print("Transcript spans:", nonzero_positions.min().item(), "to", nonzero_positions.max().item())

    if i == 4:
        break

  0%|          | 0/38 [00:00<?, ?it/s]/Users/douglasflorizone/Summer 2025 Research/Code/Spliceformer/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 11%|█         | 4/38 [00:00<00:00, 64.33it/s]

Transcript spans: 20000 to 44999
Transcript spans: 15000 to 44999
Transcript spans: 10000 to 44999
Transcript spans: 5000 to 44999
Transcript spans: 0 to 44999


In [165]:
def one_hot_decode(batch_features):
    one_hot = batch_features.squeeze(0)
    idx_to_base = ['A', 'C', 'G', 'T']
    base_indices = torch.argmax(one_hot, dim=0)
    sequence = ''.join(idx_to_base[i] for i in base_indices.tolist())
    return sequence

In [7]:
def default_inner_dict():
    return defaultdict(list)

In [207]:
def seq_finder(batch_features,targets,path):
    with open(path, 'rb') as f:
        grad_cam_dict = pickle.load(f)
    grad_cam_full = []
    acceptor_idxs = np.where(targets[1,:]==1)
    donor_idxs = np.where(targets[2,:]==1)
    splice_idxs = np.where(targets[1:,:]==1)
    start,end = 0,45000
    start_shift = 20000
    shift = 20000
    nucleotides = one_hot_decode(batch_features)
    for label,key in zip(splice_idxs[0],splice_idxs[1]):
        for i,layer in enumerate(grad_cam_dict[key]):
            grad_cam_key = torch.stack(grad_cam_dict[key][layer])
            grad_cam_key = torch.mean(grad_cam_key,dim=0)
            relu = torch.nn.ReLU()
            grad_cam_key = relu(grad_cam_key)
            grad_cam_key = grad_cam_key.detach().numpy()
            if grad_cam_key.max()  != np.float32(0):
                grad_cam_key /= grad_cam_key.max()
            grad_cam_full.append(grad_cam_key)

        differences = grad_cam_full[0] - grad_cam_full[3]
        positives = np.where(differences>0.6)[0]
        negatives = key
        relevant_pos_regions = []
        relevant_neg_regions = []
        relevant_neg_regions.append(nucleotides[negatives-100:negatives+100])
        previous_end = 0
        for positive in positives:
            start = max(0, positive-100)
            end = min(len(nucleotides), positive + 100) 
            if start < previous_end:
                previous_end = end
                continue
            previous_end = end
            relevant_pos_regions.append(nucleotides[start:end])
        grad_cam_full = []
        break

    return relevant_pos_regions, relevant_neg_regions
        # print(differences.tolist())
            # if grad_cam_key.max()  != np.float32(0):
            #     grad_cam_key /= grad_cam_key.max()
            # if i==0:
            #     peak_idxs = np.where(grad_cam_key>0.6)
            #     peaks = grad_cam_key[peak_idxs]
            #     sites = grad_cam_key[key]
            # elif i==3:
            #     differences = grad_cam_key[peak_idxs] - peaks
            #     differences = differences[differences<0]

In [208]:
pos_sequences, neg_sequences = seq_finder(batch_features,targets,path)


In [211]:
sequences_data = []
for i in range(len(neg_sequences)):
    sequences_data.append([f'seq{i}', neg_sequences[i]])

if os.path.exists('../Results/Grad-CAM/neg_test_sequences.fasta'):
    os.remove('../Results/Grad-CAM/neg_test_sequences.fasta')
with open('../Results/Grad-CAM/neg_test_sequences.fasta', 'w') as f:
    for id, sequence in sequences_data:
        f.write(f'>{id}\n')
        f.write(f'{sequence}\n')

# fa = pyfastx.Fasta('../Results/Grad-CAM/test_sequences.fasta')
# print(fa['seq15'])

In [212]:
for seq in pyfastx.Fasta('../Results/Grad-CAM/neg_test_sequences.fasta'):
    print(f"Sequence: {seq}")

Sequence: CATAATTTTCCATATGCCAGAAAAGTTGAATAGTATCAGATTCCAAATCTGTATGGAGACCAAATCAAGTGAATATCTGTTCCTCCTCTCTTTATTTTAGCTGGACCAGACCAATTTTGAGGAAAGGATACAGACAGCGCCTGGAATTGTCAGACATATACCAAATCCCTTCTGTTGATTCTGCTGACAATCTATCTGAA
